In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

#Cargar Data Set

In [4]:
df = pd.read_csv(
    "Telco_customer_churn.csv",
    sep=";"
)

print(df.shape)

(7043, 33)


#Seleccionar variables

In [5]:
columnas = [
    "Tenure Months",
    "Monthly Charges",
    "Total Charges",
    "Churn Label"
]

df_ml = df[columnas].copy()

print(df_ml.shape)

(7043, 4)


#Transformar variable objetivo

In [6]:
df_ml["Churn Label"] = df_ml["Churn Label"].map({
    "Yes": 1,
    "No": 0
})

#Convertir Total Charges

In [7]:
df_ml["Total Charges"] = pd.to_numeric(
    df_ml["Total Charges"],
    errors="coerce"
)

#Eliminar nulos

In [8]:
df_ml = df_ml.dropna()

print(df_ml.shape)

(7032, 4)


#Crear X e y

In [9]:
X = df_ml[
    [
        "Tenure Months",
        "Monthly Charges",
        "Total Charges"
    ]
]

y = df_ml["Churn Label"]

#Dividir entrenamiento y prueba

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(5625, 3)
(1407, 3)


#Escalar datos

In [11]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

#Crear la red neuronal

In [12]:
modelo_nn = Sequential()

modelo_nn.add(
    Dense(
        16,
        activation="relu",
        input_shape=(3,)
    )
)

modelo_nn.add(
    Dense(
        8,
        activation="relu"
    )
)

modelo_nn.add(
    Dense(
        1,
        activation="sigmoid"
    )
)

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


#Compilar el modelo

In [13]:
modelo_nn.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [14]:
modelo_nn.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 16)             │            64 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 209 (836.00 B)

 Trainable params: 209 (836.00 B)

 Non-trainable params: 0 (0.00 B)

#Entrenar

In [15]:
historial = modelo_nn.fit(
    X_train_scaled,
    y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

Epoch 1/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.6542 - loss: 0.6241 - val_accuracy: 0.7502 - val_loss: 0.5175
Epoch 2/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.7660 - loss: 0.4935 - val_accuracy: 0.7956 - val_loss: 0.4528
Epoch 3/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.7736 - loss: 0.4654 - val_accuracy: 0.7911 - val_loss: 0.4402
Epoch 4/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7751 - loss: 0.4587 - val_accuracy: 0.7956 - val_loss: 0.4358
Epoch 5/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7773 - loss: 0.4560 - val_accuracy: 0.7964 - val_loss: 0.4336
Epoch 6/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7798 - loss: 0.4543 - val_accuracy: 0.8018 - val_loss: 0.4326
Epoch 7/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7809 - loss: 0.4543 - val_accuracy: 0.8053 - val_loss: 0.4312
Epoch 8/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7791 - loss: 0.4538 - val_accuracy: 0.

#Evaluar

In [16]:
loss, accuracy = modelo_nn.evaluate(
    X_test_scaled,
    y_test
)

print(
    "Accuracy Red Neuronal:",
    round(accuracy,4)
)

44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7953 - loss: 0.4563
Accuracy Red Neuronal: 0.7953


#Comparacion

In [17]:
accuracy_logistica = 79.25
accuracy_red = round(accuracy * 100, 2)

print("Regresión Logística:", accuracy_logistica, "%")
print("Red Neuronal:", accuracy_red, "%")

Regresión Logística: 79.25 %
Red Neuronal: 79.53 %


## Conclusiones US-07

### Implementación de la Red Neuronal

Se desarrolló una red neuronal utilizando **TensorFlow/Keras** para la predicción de abandono de clientes.

La arquitectura implementada estuvo compuesta por:

- **Capa de entrada:** 16 neuronas
- **Capa oculta:** 8 neuronas
- **Capa de salida:** 1 neurona con función de activación **Sigmoid**

Los datos fueron normalizados mediante **StandardScaler** antes del entrenamiento.

---

### Resultados obtenidos

Se comparó el desempeño de la red neuronal contra el modelo de **Regresión Logística** desarrollado previamente.

| Modelo | Accuracy |
|--------|----------|
| Regresión Logística | 79.25 % |
| Red Neuronal | 79.18 % |

---

### Análisis de resultados

- La diferencia entre ambos modelos fue **mínima**.
- La red neuronal obtuvo un rendimiento muy similar al de la Regresión Logística, sin mostrar una mejora significativa en la precisión de las predicciones.
- Esto sugiere que las variables utilizadas contienen patrones relativamente **simples** que pueden ser capturados adecuadamente mediante modelos tradicionales de Machine Learning.

---

### Conclusiones

- ✅ Se implementó exitosamente una red neuronal utilizando **TensorFlow/Keras**.
- ✅ El modelo fue entrenado y evaluado utilizando datos históricos de clientes.
- ✅ Se realizó una comparación contra un modelo tradicional de **Regresión Logística**.
- ⚠️ La Regresión Logística obtuvo un rendimiento **ligeramente superior**.
- 📌 Los resultados demuestran que, para este conjunto de datos, una red neuronal **no necesariamente mejora** el desempeño predictivo.

## Sistema de Scoring de Riesgo de Abandono

El sistema de scoring convierte la probabilidad de abandono de un cliente en tres niveles de riesgo: Bajo, Medio y Alto.

Se utilizan umbrales iniciales para facilitar la interpretación de los resultados:
- Bajo: probabilidad menor al 30%.
- Medio: probabilidad desde 30% hasta menos del 60%.
- Alto: probabilidad igual o superior al 60%.

In [18]:
def clasificar_riesgo(probabilidad):
    """
    Clasifica el riesgo de abandono de un cliente.

    Parámetro:
        probabilidad: valor entre 0 y 1.

    Retorna:
        Nivel de riesgo: Bajo, Medio o Alto.
    """

    if not 0 <= probabilidad <= 1:
        raise ValueError("La probabilidad debe estar entre 0 y 1.")

    if probabilidad < 0.30:
        return "Bajo"
    elif probabilidad < 0.60:
        return "Medio"
    else:
        return "Alto"

In [19]:
probabilidades_prueba = [0.15, 0.45, 0.85]

for probabilidad in probabilidades_prueba:
    riesgo = clasificar_riesgo(probabilidad)

    print(
        f"Probabilidad: {probabilidad:.0%} | "
        f"Nivel de riesgo: {riesgo}"
    )

Probabilidad: 15% | Nivel de riesgo: Bajo
Probabilidad: 45% | Nivel de riesgo: Medio
Probabilidad: 85% | Nivel de riesgo: Alto


In [20]:
assert clasificar_riesgo(0.00) == "Bajo"
assert clasificar_riesgo(0.29) == "Bajo"
assert clasificar_riesgo(0.30) == "Medio"
assert clasificar_riesgo(0.59) == "Medio"
assert clasificar_riesgo(0.60) == "Alto"
assert clasificar_riesgo(1.00) == "Alto"

print("Todas las pruebas de scoring fueron exitosas.")

Todas las pruebas de scoring fueron exitosas.
